# METIS All-in-One Workflow Notebook

This notebook demonstrates the complete end-to-end workflow for the **METIS** instrument at the ELT, covering:

1. **Simulation** — generate synthetic raw FITS frames with `METIS_Simulations` (ScopeSim backend)
2. **Pipeline Reduction** — reduce raw frames with a `METIS_Pipeline` recipe (`MetisDetDark`)
3. **Archive Ingestion** — ingest raw frames into `MetisWISE` (skeleton; server connectivity required)

The example uses a **LM-band dark calibration** as the simplest possible end-to-end case:
- Mode: `img_lm` (LM-band direct imaging)
- Observation type: dark (closed shutter, no sky signal)
- Number of simulated frames: 3
- Recipe: `MetisDetDark` → produces a master dark frame

---

### Repository layout

```
MTR/
├── METIS_Simulations/          ScopeSim-based synthetic observation generator
│   ├── YAML/ESO/               Observation block YAML templates
│   └── metis_simulations/      Python package (runSimulationBlock, setupSimulations, …)
├── METIS_Pipeline/             ESO CPL/EDPS data reduction pipeline
│   └── metisp/pymetis/src/     Python recipe implementations
├── inst_pkgs/                  ScopeSim instrument package cache (METIS, ELT, Armazones)
├── output/sim/                 Simulated raw FITS files (written by Section 1)
├── output/pipeline/            Reduced data products (written by Section 3)
└── all-in-one.ipynb            ← this notebook
```

---
## Section 0 — Environment Bootstrap

### Why this section must run first

The `metis_simulations` package reads the environment variable `DEFAULT_IRDB_LOCATION` **at import time** to locate the ScopeSim instrument packages (METIS, ELT, Armazones). If this variable is not set before the first `import scopesim`, ScopeSim falls back to a default path that will likely not exist on this machine, causing a runtime error when the simulation tries to load instrument data.

The same applies to `PYCPL_RECIPE_DIR` and `PYESOREX_PLUGIN_DIR`, which tell the CPL/PyEsoRex layer where to find the compiled and Python-based METIS recipes.

**Rule:** always run this cell before any other cell in the notebook.

### Path layout

All paths are derived from the MTR root so the notebook is portable across machines:

| Variable | Points to |
|---|---|
| `SIMS_ROOT` | `METIS_Simulations/` — Python package + YAML templates |
| `PIPE_ROOT` | `METIS_Pipeline/metisp/pymetis/src/` — pipeline Python packages |
| `RECIPE_DIR` | `METIS_Pipeline/metisp/pyrecipes/` — compiled C and Python recipe plugins |
| `INST_PKGS` | `inst_pkgs/` — ScopeSim instrument package cache |

In [1]:
import os
import sys
from pathlib import Path

auto_locate = True
# Automatically locate the MTR root relative to this notebook's location
if auto_locate:
    import json 
    MTR_ROOT = Path(json.loads(Path(sys.argv[-1]).read_bytes())['jupyter_session']).resolve().parent
else:
    # Fallback for Jupyter: set MTR_ROOT manually if the auto-detection fails
    MTR_ROOT = Path('/home/k_man/MTR')

SIMS_ROOT  = MTR_ROOT / 'METIS_Simulations'
PIPE_ROOT  = MTR_ROOT / 'METIS_Pipeline' / 'metisp' / 'pymetis' / 'src'
RECIPE_DIR = MTR_ROOT / 'METIS_Pipeline' / 'metisp' / 'pyrecipes'
INST_PKGS  = MTR_ROOT / 'inst_pkgs'

# Add both package roots to the Python path
for p in (str(SIMS_ROOT), str(PIPE_ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

# Environment variables consumed by CPL / PyEsoRex / ScopeSim at import time
os.environ.setdefault('DEFAULT_IRDB_LOCATION', str(INST_PKGS))
os.environ.setdefault('PYCPL_RECIPE_DIR',       str(RECIPE_DIR))
os.environ.setdefault('PYESOREX_PLUGIN_DIR',    str(RECIPE_DIR))
os.environ.setdefault('PYESOREX_MSG_LEVEL',     'debug')
os.environ.setdefault('PYESOREX_LOG_LEVEL',     'debug')

print(f'MTR_ROOT   : {MTR_ROOT}')
print(f'SIMS_ROOT  : {SIMS_ROOT}')
print(f'PIPE_ROOT  : {PIPE_ROOT}')
print(f'INST_PKGS  : {INST_PKGS}')

MTR_ROOT   : /home/k_man/MTR
SIMS_ROOT  : /home/k_man/MTR/METIS_Simulations
PIPE_ROOT  : /home/k_man/MTR/METIS_Pipeline/metisp/pymetis/src
INST_PKGS  : /home/k_man/MTR/inst_pkgs


### ScopeSim instrument packages

ScopeSim needs a local copy of the METIS and ELT instrument packages to model the telescope and detector optics. These are downloaded once and cached under `inst_pkgs/`. The cell below imports ScopeSim (now safe because `DEFAULT_IRDB_LOCATION` is already set), overrides the runtime config to point at the same cache directory, and triggers an automatic download if the packages are absent.

In [2]:
(INST_PKGS / 'METIS').is_dir()

True

In [3]:
import scopesim as sim

# Runtime override — must also be set here in addition to the env var
# because some ScopeSim code paths read from rc.__config__ directly
# sim.rc.__config__['!SIM.file.local_packages_path'] = str(INST_PKGS)
sim.link_irdb(str(INST_PKGS))

if not (INST_PKGS / 'METIS').is_dir():
    print(f'Instrument packages not found at {INST_PKGS}. Downloading …')
    INST_PKGS.mkdir(parents=True, exist_ok=True)
    sim.download_packages('METIS', release='2026-02-18')
    sim.download_packages('ELT',   release='2025-10-26')
else:
    print(f'Instrument packages found at {INST_PKGS}')

Instrument packages found at /home/k_man/MTR/inst_pkgs


---
## Section 1 — Simulation

### What METIS_Simulations does

`METIS_Simulations` is a Python package that wraps [ScopeSim](https://scopesim.readthedocs.io/) to generate realistic synthetic raw FITS files for METIS. It models:

- **Optical train** — ELT primary, secondary, tertiary mirrors + METIS relay and detector optics
- **Detector physics** — read noise, dark current, non-linearity, persistence
- **Sky background** — thermal emission, atmospheric transmission (via SkyCalc)
- **Astronomical sources** — point sources, extended targets, or an empty dark sky

The simulation is driven entirely by a **YAML observation block file**.

---

### The YAML observation block format

Each top-level key in the YAML defines one *recipe* (a single exposure or a sequence of identical exposures):

```yaml
DARK_LM_RAW:
  do.catg: DARK_LM_RAW      # ESO archive classification tag
  mode: "img_lm"            # ScopeSim instrument mode
  source:
    name: empty_sky          # no signal — closed shutter dark
    kwargs: {}
  properties:
    dit: 1.                  # detector integration time (seconds)
    ndit: 1                  # number of co-added reads per exposure
    filter_name: "closed"    # closed shutter → dark frame
    catg: "CALIB"            # → DPR.CATG FITS header keyword
    tech: "IMAGE,LM"         # → DPR.TECH FITS header keyword
    type: "DARK"             # → DPR.TYPE FITS header keyword
    nObs: 3                  # number of independent exposures to simulate
```

The `DPR.*` header keywords written by `updateHeaders()` are what the pipeline later uses to classify frames into *reduction tags* (see Section 2).

---

### `runSimulationBlock` — the entry point

```python
runSimulationBlock(yamlFiles, params, args)
```

| Argument | Type | Purpose |
|---|---|---|
| `yamlFiles` | `list[str]` | One or more YAML observation block files to process in sequence |
| `params` | `dict` | Runtime parameters (output directory, calibration options, etc.) |
| `args` | `list` | Command-line override arguments (empty list when called programmatically) |

Key `params` entries:

| Key | Meaning |
|---|---|
| `outputDir` | Directory where FITS files are written |
| `doCalib` | Number of auto-generated calibration frames (0 = none beyond what the YAML specifies) |
| `doStatic` | Generate static calibration prototypes (PERSISTENCE_MAP, ATM_PROFILE, …) |
| `nCores` | CPU cores for parallel simulation (multiprocessing.Pool) |
| `testRun` | If `True`, parse YAML and validate headers only — do not run ScopeSim |
| `startMJD` | Override observation start time (`'YYYY-MM-DD HH:MM:SS'` or `None`) |

In [7]:
from metis_simulations.runSimulationBlock import runSimulationBlock

YAML_FILE = SIMS_ROOT / 'YAML' / 'ESO' / 'darkLM.yaml'
SIM_OUT   = MTR_ROOT / 'output' / "20260420T211221" / 'sim'
SIM_OUT.mkdir(parents=True, exist_ok=True)

params = dict(
    outputDir = str(SIM_OUT),
    small     = False,   # True → 32×32 detector (fast CI testing only)
    doStatic  = False,   # skip static calibration prototype generation
    doCalib   = 0,       # no additional auto-generated calibration frames
    sequence  = False,   # do not chain observation start times across YAML files
    startMJD  = None,    # use the date embedded in the YAML
    calibFile = None,
    nCores    = 1,
    testRun   = False,
)

print(f'Simulating dark frames from {YAML_FILE.name} …')
#runSimulationBlock([str(YAML_FILE)], params, [])

dark_files = sorted(SIM_OUT.glob('METIS.DARK_LM_RAW.*.fits'))
print(f'\nProduced {len(dark_files)} dark frame(s):')
for f in dark_files:
    print(f'  {f.name}')

Simulating dark frames from darkLM.yaml …

Produced 3 dark frame(s):
  METIS.DARK_LM_RAW.2027-01-25_00_01_09.fits
  METIS.DARK_LM_RAW.2027-01-25_00_01_11.fits
  METIS.DARK_LM_RAW.2027-01-25_00_01_13.fits


---
## Section 2 — Frameset Construction

### What is a FrameSet?

The METIS pipeline is built on the **ESO Common Pipeline Library (CPL)**. CPL's fundamental input abstraction is a `cpl.ui.FrameSet` — essentially a list of `(filename, classification_tag)` pairs. The classification tag tells each recipe what role a file plays in the reduction (e.g., raw science frame, calibration reference, bad pixel mask).

Historically, FrameSets were always loaded from a plain-text **SOF file** (Set of Frames):

```
/path/to/METIS.DARK_LM_RAW.2027-01-25_00_00_00.fits  DARK_2RG_RAW
/path/to/METIS.DARK_LM_RAW.2027-01-25_00_01_01.fits  DARK_2RG_RAW
/path/to/METIS.DARK_LM_RAW.2027-01-25_00_02_02.fits  DARK_2RG_RAW
```

The `from_tags` and `from_filenames` helpers in `pymetis.engine.core.functions.frameset` provide a Pythonic API that constructs the same `cpl.ui.FrameSet` programmatically without writing a file.

---

### Frame classification: `DARK_LM_RAW` vs `DARK_2RG_RAW`

There are two tag namespaces in play here, which is a common source of confusion:

| Tag | Used by | Meaning |
|---|---|---|
| `DARK_LM_RAW` | ESO archive, EDPS workflow scheduler | The `do.catg` value from the YAML; written into `DRS.CATG` in the FITS header; used by the EDPS orchestrator to schedule which workflow task to run |
| `DARK_2RG_RAW` | CPL recipe `InputSet` | The SOF classification tag that the recipe's `Dark2rgRaw` input class matches against; derived from the name template `DARK_{detector}_RAW` with `detector='2RG'` |

The mapping between the two is encoded in `run_metis.py`'s `DPR_TO_TAG` table:
```python
(DPR.CATG='CALIB', DPR.TYPE='DARK', DPR.TECH='IMAGE,LM')  →  'DARK_2RG_RAW'
```

When building the frameset programmatically, use the **recipe tag** (`DARK_2RG_RAW`), not the archive tag.

---

### Optional calibration inputs for MetisDetDark

The recipe also accepts optional calibration reference files. All are optional — the recipe degrades gracefully if they are absent:

| Tag | Purpose |
|---|---|
| `LINEARITY_2RG` | Non-linearity correction map |
| `PERSISTENCE_MAP` | Persistence correction map |
| `GAIN_MAP_2RG` | Per-pixel gain map |
| `BADPIX_MAP_2RG` | Static bad pixel mask |

To include them, extend the `from_tags` call:
```python
frameset = from_tags(
    DARK_2RG_RAW     = [str(f) for f in dark_files],
    LINEARITY_2RG    = ['/path/to/LINEARITY_2RG.fits'],
    PERSISTENCE_MAP  = ['/path/to/PERSISTENCE_MAP.fits'],
)
```

In [18]:
from pymetis.engine.core.functions.frameset import from_tags

frameset = from_tags(
    DARK_2RG_RAW=[str(f.absolute()) for f in dark_files],
    # Add optional calibration inputs here when available:
    # LINEARITY_2RG   = ['/path/to/LINEARITY_2RG.fits'],
    # PERSISTENCE_MAP = ['/path/to/PERSISTENCE_MAP.fits'],
    # GAIN_MAP_2RG    = ['/path/to/GAIN_MAP_2RG.fits'],
)

print(f'Frameset contents:')
for frame in frameset:
    print(f'{Path(frame.file).name}    {frame.tag}')

Frameset contents:
METIS.DARK_LM_RAW.2027-01-25_00_01_09.fits    DARK_2RG_RAW
METIS.DARK_LM_RAW.2027-01-25_00_01_11.fits    DARK_2RG_RAW
METIS.DARK_LM_RAW.2027-01-25_00_01_13.fits    DARK_2RG_RAW


---
## Section 3 — Pipeline Reduction

### What is MetisDetDark?

`MetisDetDark` is the METIS pipeline recipe that creates a **master dark frame** from a set of raw dark exposures. A master dark captures the thermal and electronic baseline signal of the detector — the counts that accumulate even with the shutter closed. It is subtracted from all subsequent science and calibration frames before any other processing.

The recipe is implemented in `METIS_Pipeline/metisp/pymetis/src/pymetis/instruments/metis/recipes/metis_det_dark.py` and supports all three METIS detector systems:

| Detector | Tag | Mode |
|---|---|---|
| 2RG (H2RG, LM band) | `DARK_2RG_RAW` | `img_lm` |
| GEO (GeoSnap, N band) | `DARK_GEO_RAW` | `img_n` |
| IFU (4× H2RG) | `DARK_IFU_RAW` | `lms` |

This notebook uses the 2RG (LM-band) variant.

---

### What MetisDetDark computes

1. Load each raw dark frame per detector
2. Apply gain, persistence, and non-linearity corrections (when calibration inputs are provided)
3. Combine frames using the selected stacking method (default: `average`)
4. Estimate read noise from the difference of two frames
5. Sigma-clip to identify and mask hot, cold, and bad pixels
6. Collect quality control (QC) parameters: median, RMS, bad pixel counts
7. Write a multi-extension FITS product:

```
HDU 0  PRIMARY   (empty)
HDU 1  DET1.SCI  combined dark (science data)
HDU 2  DET1.ERR  propagated uncertainty
HDU 3  DET1.DQ   data quality / bad pixel mask
```

---

### Recipe parameters

The second argument to `.run()` is a `dict` of parameter overrides. The only parameter exposed by `MetisDetDark` is:

| Parameter | Default | Options |
|---|---|---|
| `metis_det_dark.stacking.method` | `average` | `add`, `average`, `median`, `sigclip` |

Example: `MetisDetDark().run(frameset, {'metis_det_dark.stacking.method': 'median'})`

---

### Working directory

The CPL layer writes output FITS products to the **current working directory** (a PyEsoRex convention inherited from the C-based pipeline infrastructure). We temporarily `chdir` to `output/pipeline/` and restore the original directory in a `finally` block so the notebook's working directory is not permanently changed.

In [19]:
from pymetis.instruments.metis.recipes import MetisDetDark

PIPE_OUT = MTR_ROOT / 'output' / 'pipeline'
PIPE_OUT.mkdir(parents=True, exist_ok=True)

_orig_cwd = Path.cwd()
os.chdir(PIPE_OUT)
print(f'Working directory → {PIPE_OUT}')

try:
    result_frameset = MetisDetDark().run(frameset, {})
finally:
    os.chdir(_orig_cwd)

print('\nPipeline products:')
for frame in result_frameset:
    print(f'  [{frame.tag}]  {frame.filename}')

Working directory → /home/k_man/MTR/output/pipeline
[ INFO  ] Promoting ProductSet with {'detector': '2RG'}
[ INFO  ] Promoting Qc with {'detector': '2RG'}
[ INFO  ] Loading raw dark data
[ INFO  ] Loading Dark2rgRaw items
[ INFO  ]  - loading input frame #0: '/home/k_man/MTR/output/20260420T211221/sim/METIS.DARK_LM_RAW.2027-01-25_00_01_09.fits'
[ INFO  ]  - loading input frame #1: '/home/k_man/MTR/output/20260420T211221/sim/METIS.DARK_LM_RAW.2027-01-25_00_01_11.fits'
[ INFO  ]  - loading input frame #2: '/home/k_man/MTR/output/20260420T211221/sim/METIS.DARK_LM_RAW.2027-01-25_00_01_13.fits'
[ INFO  ] Input Items loaded: [<DataItem DARK_2RG_RAW>, <DataItem DARK_2RG_RAW>, <DataItem DARK_2RG_RAW>]
[ INFO  ] Processing detector 1
[ INFO  ] Loading Dark2rgRaw items
[ INFO  ] Loading extension 'DET1.DATA' from multiple frames [(file=/home/k_man/MTR/output/20260420T211221/sim/METIS.DARK_LM_RAW.2027-01-25_00_01_09.fits, tag=DARK_2RG_RAW, group=FrameGroup.RAW, level=FrameLevel.FINAL, type=Frame

AttributeError: 'NoneType' object has no attribute 'file'

### Inspect the master dark

Use `astropy.io.fits` to quickly inspect the structure of the output product.

In [ ]:
from astropy.io import fits

master_dark_files = sorted(PIPE_OUT.glob('MASTER_DARK_2RG*.fits'))
if master_dark_files:
    with fits.open(master_dark_files[0]) as hdul:
        hdul.info()
        print(f'\nDET1.SCI shape: {hdul["DET1.SCI"].data.shape}')
        print(f'DET1.SCI mean : {hdul["DET1.SCI"].data.mean():.4f} ADU')
else:
    print('No master dark found — check pipeline output above for errors.')

---
## Section 4 — Archive Ingestion (MetisWISE)

### What is MetisWISE?

**MetisWISE** is the bulk storage and metadata archive for METIS data products. It is built on the **ADARI/AWE framework** (Astronomical Wide-field imager Environment), which provides:

- A relational database of data item metadata (auto-populated from FITS headers)
- A file storage back-end for the raw and reduced FITS files
- A Python query API based on attribute comparisons (`Raw.det_dit < 5`)

### Two-step commit pattern

Ingestion always follows a two-step sequence:

1. **`.store()`** — uploads the FITS file bytes to the bulk storage back-end
2. **`.commit()`** — persists the parsed FITS header metadata to the relational database

Calling `.commit()` without `.store()` raises a warning and creates a dangling database record with no associated file. The two steps are intentionally separate to allow atomic transactions.

### Data classes

Each frame type has a corresponding Python class generated from the DRLD (Data Reduction Library Description). For raw LM darks:

```python
from metiswise.main.raw import DARK_2RG_RAW as Dark2rgRawClass
```

The class exposes every standard FITS keyword as a typed Python attribute (e.g., `raw_obj.det_dit`, `raw_obj.mjdobs`, `raw_obj.dpr_tech`) mapped from FITS header names via the `persistent()` descriptor.

### Query API

```python
# All raws of this type in the archive
all_raws = Dark2rgRawClass.select_all()

# Filter by header value
short_darks = Dark2rgRawClass.det_dit < 5      # returns a list
my_dark = short_darks[0]
my_dark.retrieve()                              # pull the file back from storage
```

> **Status:** the MetisWISE server is not currently reachable. The code below is a ready-to-activate skeleton — uncomment it once connectivity is available.

In [ ]:
# ── MetisWISE raw ingestion ───────────────────────────────────────────────────
# Uncomment when the MetisWISE server is reachable.
# Requires the metiswise package on PYTHONPATH.

# from metiswise.main.aweimports import *   # imports Raw, DataItem, context, …
# from metiswise.main.raw import DARK_2RG_RAW as Dark2rgRawClass
#
# print(f'Existing {Dark2rgRawClass.__name__} entries in archive: '
#       f'{len(Dark2rgRawClass.select_all())}')
#
# for fits_file in dark_files:
#     raw_obj = Dark2rgRawClass(str(fits_file))
#     print(f'  {fits_file.name}  DIT={raw_obj.det_dit} s  NDIT={raw_obj.det_ndit}')
#     raw_obj.store()    # upload file bytes to bulk storage
#     raw_obj.commit()   # persist FITS header metadata to the database
#
# # Query example — retrieve all short darks
# results = Dark2rgRawClass.det_dit < 5
# print(f'\nFound {len(results)} archive entries with DIT < 5 s')
# for r in results:
#     print(f'  {r.filename}  MJD-OBS={r.mjdobs:.5f}')

print('MetisWISE ingestion skipped (server not yet reachable).')
print('Uncomment the code above when connectivity is available.')

---
## Next steps

This notebook covers the minimal dark-only chain. To extend it:

| Extension | What to change |
|---|---|
| Run a different observation mode | Replace `darkLM.yaml` with another YAML from `METIS_Simulations/YAML/ESO/` and update the frame tag and recipe accordingly |
| Include calibration reference files | Add `LINEARITY_2RG`, `PERSISTENCE_MAP`, etc. to the `from_tags()` call in Section 2 |
| Run the full workflow (darks + science) | Pass multiple YAML files to `runSimulationBlock` and wire the output through the appropriate EDPS workflow task chain (see `WORKFLOW_TASK_CHAIN` in `run_metis.py`) |
| Change stacking method | Pass `{'metis_det_dark.stacking.method': 'median'}` as the second argument to `MetisDetDark().run()` |
| Enable auto-generated calibrations | Set `doCalib=3` in `params` to have `runSimulationBlock` automatically generate matching dark and flat sets |
| Activate archive ingestion | Uncomment Section 4 once the MetisWISE server is reachable |